# Welcome to the Combining FITS to Make Color Images Tutorial

### The cell below is the *ONLY* cell you need to edit in this notebook.
Enter the file name of your 4 FITS files here. 


For r, enter the file name of your FITS file appended "r_cal.fit".<br>
For g, enter the file name of your FITS file appended "g_cal.fit".<br>
For b, enter the file name of your FITS file appended "b_cal.fit".<br>
For l, enter the file name of your FITS file appended "l_cal.fit".


Make sure your FITS files are in the same folder as this Jupyter notebook.

In [ ]:
# This part defines where your FITS files are located
r_fits_file_path= ...
g_fits_file_path= ...
b_fits_file_path= ...
l_fits_file_path= ...

The following cell installs astropy, numpy, matplotlib, scikit-image, and scipy if you need it.

In [ ]:
!pip install "numpy>=1.24,<2.0" astropy matplotlib scikit-image scipy

The following cell imports various libraries and switches matplotlib's backend to an "interactive notebook mode" (%matplotlib widget) so that sliders, buttons, and live updates work inline.

In [ ]:
import matplotlib
matplotlib.use('module://ipympl.backend_nbagg')
import matplotlib.pyplot as plt
%matplotlib widget
import numpy as np
from matplotlib.widgets import Slider
from astropy.io import fits
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
from scipy.ndimage import shift

The following cell defines a "center crop" function that does the following:
1. Ensures all four images have the same shape before combining.
2. Finds the smallest height and width among all images, then center-crops the others to match.
3. Returns all cropped images in the same order as given.

This prevents misalignment or shape mismatch errors when stacking or blending images later.

In [ ]:
# Center crop function
def center_crop_to_smallest(reference, *arrays):
    shapes = [reference.shape] + [arr.shape for arr in arrays]
    min_y = min(s[0] for s in shapes)
    min_x = min(s[1] for s in shapes)

    def crop(arr):
        y, x = arr.shape
        start_y = (y - min_y) // 2
        start_x = (x - min_x) // 2
        return arr[start_y:start_y + min_y, start_x:start_x + min_x]

    cropped = [crop(reference)] + [crop(arr) for arr in arrays]
    return tuple(cropped)

The following cell is where the FITS files are combined, normalized, and creates the LRGB image based on the inputs for red (rf), green (gf), blue (bf), saturation (sat), Contrast (contrast), Exposure (exposure), Luminance (lum_scale), and Gamma (gamma). There are comments throughout this cell that explains what each part of the code is doing.

In [ ]:
# This part load your FITS images
r = fits.getdata(r_fits_file_path)
g = fits.getdata(g_fits_file_path)
b = fits.getdata(b_fits_file_path)
l = fits.getdata(l_fits_file_path)

# This center-crops images to match shape
r, g, b, l = center_crop_to_smallest(r, g, b, l)

# This function normalizes the data by removing any NaNs, 
# shifts the data so the minimum value is 0,
# and rescales intensities to between 0 and 1.
def normalize_percentile(data, pmin=0.5, pmax=99.5):
    data = np.nan_to_num(data)
    low = np.percentile(data, pmin)
    high = np.percentile(data, pmax)
    # avoid division by zero
    if high - low == 0:
        return np.clip(data - low, 0, 1)
    arr = (data - low) / (high - low)
    return np.clip(arr, 0, 1)

# This runs each FITS file through the normalization function
r_norm = normalize_percentile(r)
g_norm = normalize_percentile(g)
b_norm = normalize_percentile(b)
l_norm = normalize_percentile(l)

#This function builds the LRGB based on the image parameters
#(red, green, blue, luminance, saturation, contrast, exposure, and gamma).
def make_lrgb(rf, gf, bf, sat, contrast, exposure, lum_scale, gamma):
    #This stacks the normalized FITS data
    rgb = np.dstack([
        np.clip(r_norm * rf, 0, 1),
        np.clip(g_norm * gf, 0, 1),
        np.clip(b_norm * bf, 0, 1)
    ])
    #This converts the image to the Hue, Saturation, Value color space to easily adjust saturation.
    hsv = mcolors.rgb_to_hsv(rgb)
    hsv[..., 1] *= sat

    #This uses luminance to control brightness and contrast.
    #lum_scale scales the luminance channel
    #contrast stretches pixel values around the mean
    #exposure brightens or darkens the image globally
    #gamma applies nonlinear brightness correction
    v = l_norm.copy() * lum_scale
    mean = np.mean(v)
    v = (v - mean) * contrast + mean
    v *= exposure
    v = np.clip(v, 0, 1)
    v = v ** (1.0 / gamma)

    #This replaces the brightness channel with the adjusted luminance value,
    #and then converts the images back to LRGB for displaying.
    hsv[..., 2] = np.clip(v, 0, 1)
    return np.clip(mcolors.hsv_to_rgb(hsv), 0, 1)

#This creates a widget that allows you to adjust the image parameters
#(red, green, blue, luminance, saturation, contrast, exposure, and gamma)
r_input = widgets.FloatText(value=1.2, description='Red:', step=0.1)
g_input = widgets.FloatText(value=1.0, description='Green:', step=0.1)
b_input = widgets.FloatText(value=0.9, description='Blue:', step=0.1)
sat_input = widgets.FloatText(value=1.2, description='Saturation:', step=0.1)
con_input = widgets.FloatText(value=1.15, description='Contrast:', step=0.1)
exp_input = widgets.FloatText(value=1.3, description='Exposure:', step=0.1)
l_input = widgets.FloatText(value=1.3, description='Luminance:', step=0.1)
gamma_input = widgets.FloatText(value=1.4, description='Gamma:', step=0.1)

apply_button = widgets.Button(description="Apply Changes", button_style='primary')
save_button = widgets.Button(description="Save Image", button_style='success')
output_label = widgets.Label("")

#This creates UI elements (input boxes and buttons) for all image parameters
ui = widgets.VBox([
    r_input, g_input, b_input,
    sat_input, con_input, exp_input,
    l_input, gamma_input,
    widgets.HBox([apply_button, save_button]),
    output_label
])
#This displays the widget
display(ui)

#This displays the image. The first image you see will be based on the default
#values for the image parameters.
fig, ax = plt.subplots(figsize=(8, 8))
current_image = make_lrgb(1.2, 1, 0.9, 1.2, 1.15, 1.3, 1.3, 1.4)
img_display = ax.imshow(current_image, origin='upper')
ax.set_title("LRGB Composite Image")
plt.show()

#This function pulls the new parameter values from the widgets and updates 
#the plot without reloading or recreating the figure.
# --------------------
def update_image(change=None):
    rf = r_input.value
    gf = g_input.value
    bf = b_input.value
    sat = sat_input.value
    contrast = con_input.value
    exposure = exp_input.value
    lum_scale = l_input.value
    gamma = gamma_input.value

    global current_image
    current_image = make_lrgb(rf, gf, bf, sat, contrast, exposure, lum_scale, gamma)
    img_display.set_data(current_image)
    fig.canvas.draw_idle()

#This function converts the image to a png and saves it in the directory you're working in.
def save_image(change=None):
    img_8bit = (current_image * 255).astype(np.uint8)
    img_pil = Image.fromarray(img_8bit)
    img_pil.save("lrgb_output.png")
    output_label.value = "✅ Image saved as lrgb_output.png"

#This hooks up the button clicks with the two functions
apply_button.on_click(update_image)
save_button.on_click(save_image)
